# Kaggle Training: LogFilter Tier-2 Transformer

This notebook fine-tunes `cisco-ai/SecureBERT2.0-base` as the tier-2 log-window classifier. Tier 1 remains the fast XGBoost bag-of-events model; tier 2 consumes reconstructed text windows for higher-fidelity decisions in the cascade.

Expected runtime on Kaggle Free GPU (T4 x2) is under the 9h session limit for the sampled verification run; full two-epoch training should be run only after the sampled job succeeds. Enable a GPU accelerator before running training cells.


## 1. Locate the repo

Run this notebook from the project root when possible. If Kaggle places the repo inside `/kaggle/working`, the cell below tries to find it.


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

KAGGLE_WORKING = Path('/kaggle/working')
REPO_DIR = Path.cwd()

if not (REPO_DIR / 'training' / 'train.py').exists():
    matches = list(KAGGLE_WORKING.glob('*/training/train.py'))
    if matches:
        REPO_DIR = matches[0].parents[1]

if not (REPO_DIR / 'training' / 'train.py').exists():
    raise FileNotFoundError('Could not find training/train.py. Upload or clone the repo into /kaggle/working first.')

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR / 'src'))
print('Repo:', REPO_DIR)


## 2. Install training dependencies

Kaggle already includes PyTorch, so this installs only the transformer, dataset, ONNX, and metrics packages needed by `training/train_transformer.py`.


In [ ]:
%pip install -q --upgrade-strategy only-if-needed datasets evaluate optimum[onnxruntime] pandas


## 3. Attach the HDFS TraceBench dataset

The training script expects `HDFS_v3_TraceBench/preprocessed/normal_trace.csv` and `failure_trace.csv` under the repo. If the files are attached as a Kaggle dataset, this cell symlinks the Kaggle input path into the expected project location.


In [ ]:
PREPROCESSED_DIR = REPO_DIR / 'HDFS_v3_TraceBench' / 'preprocessed'

def has_trace_files(path: Path) -> bool:
    return (path / 'normal_trace.csv').exists() and (path / 'failure_trace.csv').exists()

if not has_trace_files(PREPROCESSED_DIR):
    input_root = Path('/kaggle/input')
    candidates = [p for p in input_root.rglob('preprocessed') if has_trace_files(p)]
    if not candidates:
        # Datasets may publish trace files at the top level (no nested 'preprocessed' dir).
        candidates = [p for p in input_root.rglob('*') if p.is_dir() and has_trace_files(p)]
    if not candidates:
        raise FileNotFoundError('No Kaggle input folder with normal_trace.csv and failure_trace.csv was found.')

    source = candidates[0]
    PREPROCESSED_DIR.parent.mkdir(parents=True, exist_ok=True)
    if PREPROCESSED_DIR.exists() and not PREPROCESSED_DIR.is_symlink():
        raise FileExistsError(f'{PREPROCESSED_DIR} exists but does not contain the expected trace files.')
    if not PREPROCESSED_DIR.exists():
        os.symlink(source, PREPROCESSED_DIR, target_is_directory=True)
    print('Linked dataset:', source, '->', PREPROCESSED_DIR)
else:
    print('Dataset already available:', PREPROCESSED_DIR)


## 4. Sanity-preview the text windows

Render one normal and one failure window before launching GPU training.


In [ ]:
sys.path.insert(0, str(REPO_DIR))
from training.text_dataset import build_windows

texts, labels, stats = build_windows(sample_normal=1, sample_failure=1, random_state=42)
print(json.dumps(stats, indent=2))
for text, label in zip(texts, labels):
    kind = 'normal' if label == 0 else 'failure'
    print(f'\n--- {kind} window: label={label}, chars={len(text)} ---')
    print(text[:1200])
    if len(text) > 1200:
        print(f'... [+{len(text) - 1200} more chars]')


## 5. Sampled training run (verify environment first)

This checks tokenizer/model download, GPU training, metric generation, and ONNX export before spending a full session.


In [ ]:
sample_cmd = [
    sys.executable,
    'training/train_transformer.py',
    '--sample-normal', '50000',
    '--sample-failure', '10000',
    '--epochs', '1',
    '--output-dir', 'models/tier2',
]
print(' '.join(sample_cmd))
subprocess.run(sample_cmd, check=True)


## 6. Full training run (uncomment when sampled run succeeds)

Run this only after the sampled training succeeds. It uses the full HDFS TraceBench text-window dataset for two epochs.


In [ ]:
# full_cmd = [
#     sys.executable,
#     'training/train_transformer.py',
#     '--epochs', '2',
#     '--output-dir', 'models/tier2',
# ]
# print(' '.join(full_cmd))
# subprocess.run(full_cmd, check=True)


## 7. Inspect artifacts


In [ ]:
models_dir = REPO_DIR / 'models' / 'tier2'
for path in sorted(models_dir.glob('*')):
    if path.is_file():
        print(f'{path.name}: {path.stat().st_size:,} bytes')
    elif path.is_dir():
        size = sum(p.stat().st_size for p in path.rglob('*') if p.is_file())
        print(f'{path.name}/: {size:,} bytes')

metrics_path = models_dir / 'tier2_metrics.json'
if metrics_path.exists():
    print(json.dumps(json.loads(metrics_path.read_text()), indent=2))


## 8. Package artifacts

The zip is written to `/kaggle/working/` so it appears in Kaggle output artifacts.


In [ ]:
import zipfile

zip_path = KAGGLE_WORKING / 'logfilter-tier2-artifacts.zip'
include_names = {
    'config.json',
    'model.safetensors',
    'tokenizer.json',
    'tokenizer_config.json',
    'special_tokens_map.json',
    'vocab.json',
    'merges.txt',
    'log_classifier_tier2.onnx',
    'tier2_metrics.json',
    'tier2_label_map.json',
}

with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(models_dir.rglob('*')):
        if path.is_file() and path.name in include_names:
            zf.write(path, arcname=str(path.relative_to(REPO_DIR)))

print('Artifact zip:', zip_path)


## 9. Output description + how to consume in repo

Download `/kaggle/working/logfilter-tier2-artifacts.zip` from Kaggle outputs and extract it at the repository root. The archive contains `models/tier2/config.json`, `model.safetensors`, tokenizer files, `log_classifier_tier2.onnx`, `tier2_metrics.json`, and `tier2_label_map.json`.

The ONNX file is the production-friendly tier-2 classifier export. The HuggingFace directory remains useful for further fine-tuning, evaluation, and reproducible export. Keep these artifacts under `models/tier2/` so they remain parallel to the tier-1 `models/log_classifier.onnx` artifacts.
